# Week 12 Practicum: OOP with Mini Tetris

**Program:** TSI International Undergraduate Program

**Modules:** 8 and 9

**Environment:** Google Colab or Jupyter Notebook with Python 3.10 or later

**Estimated time:** 180 minutes

Build a small retro falling-block game while learning classes, objects, methods, encapsulation, abstraction, composition, inheritance, and polymorphism.

Read [Module 8](../Resources/Modul-8.md) and [Module 9](../Resources/Modul-9.md) alongside this notebook.

## Working method

1. Run cells from top to bottom.
2. Read the complete examples and supplied code.
3. Complete the cells marked **Student work**.
4. Keep validation cells unchanged. They fail until the corresponding exercise is complete.
5. Click **Stop** before editing game classes. Rerun the class cells and recreate the interface after editing.
6. Restart the runtime and run all cells before submission.

| Stage | Minutes |
| --- | ---: |
| Setup and class examples | 20 |
| Module 8 task: create and move pieces | 35 |
| Module 9 task: inheritance and polymorphism | 30 |
| Final task: board and game collaboration | 60 |
| Play, add tests, and explain the design | 35 |
| **Total** | **180** |

The shape data, line-clearing algorithm, display, buttons, and timer are supplied. Your work focuses on object state and selected methods. Start is a button: running all cells does not automatically start a falling loop.

## 1. Check notebook controls

Colab normally includes `ipywidgets`. If the import below fails, run `%pip install ipywidgets` in a separate cell, then rerun setup.

Click **Test controls**. The message should change to **Controls ready**. This checks the browser connection before you build the game.

If controls remain unresponsive, reconnect the runtime and rerun the notebook. The gameplay controls require a live Python kernel; GitHub's notebook preview is read-only.

In [ ]:
# Supplied setup. This also stops an interface from an earlier run.
if "_week12_view" in globals():
    _week12_view["close"]()

import random
import ipywidgets as widgets
from IPython.display import display

control_probe = widgets.Button(description="Test controls")
probe_message = widgets.HTML(value="Click Test controls.")

def check_controls(button):
    probe_message.value = "<b>Controls ready</b>"

control_probe.on_click(check_controls)
display(widgets.HBox([control_probe, probe_message]))

## 2. Plan the objects

A piece contains four cells. Coordinates use `(x, y)`: x increases to the right, y increases downward.

| Class | Owns | Main responsibility |
| --- | --- | --- |
| `Piece` | kind, x, y, rotation | Describe and change a falling piece |
| `SquarePiece` | Inherited piece state | Construct an O piece and keep its orientation fixed |
| `Board` | Its own grid | Check positions, settle pieces, clear rows |
| `TetrisGame` | A board and active piece | Apply game rules and track results |

A game **has a** board and piece: composition. A square piece **is a** piece: inheritance.

The board is 10 columns by 20 rows. Seven shape definitions and their rotation states are supplied below. You do not need to derive rotations mathematically.

In [ ]:
# Supplied local cell coordinates, grouped by rotation state.
SHAPES = {
    "I": (
        ((0, 1), (1, 1), (2, 1), (3, 1)),
        ((1, 0), (1, 1), (1, 2), (1, 3)),
    ),
    "O": (
        ((0, 0), (1, 0), (0, 1), (1, 1)),
    ),
    "T": (
        ((1, 0), (0, 1), (1, 1), (2, 1)),
        ((1, 0), (1, 1), (2, 1), (1, 2)),
        ((0, 1), (1, 1), (2, 1), (1, 2)),
        ((1, 0), (0, 1), (1, 1), (1, 2)),
    ),
    "S": (
        ((1, 0), (2, 0), (0, 1), (1, 1)),
        ((1, 0), (1, 1), (2, 1), (2, 2)),
    ),
    "Z": (
        ((0, 0), (1, 0), (1, 1), (2, 1)),
        ((2, 0), (1, 1), (2, 1), (1, 2)),
    ),
    "J": (
        ((0, 0), (0, 1), (1, 1), (2, 1)),
        ((1, 0), (2, 0), (1, 1), (1, 2)),
        ((0, 1), (1, 1), (2, 1), (2, 2)),
        ((1, 0), (1, 1), (0, 2), (1, 2)),
    ),
    "L": (
        ((2, 0), (0, 1), (1, 1), (2, 1)),
        ((1, 0), (1, 1), (1, 2), (2, 2)),
        ((0, 1), (1, 1), (2, 1), (0, 2)),
        ((0, 0), (1, 0), (1, 1), (1, 2)),
    ),
}

## 3. Module 8 review: one class, two objects

A **class** defines an object's state and behavior. An **object**, or instance, is one value created from that class.

`__init__` initializes the instance. `self` refers to the instance receiving a method call. An attribute stores state; a method performs an operation.

Run the example. Only the first block changes.

In [ ]:
class BlockExample:
    def __init__(self, y=0):
        self.y = y

    def drop(self):
        self.y += 1


first_block = BlockExample()
second_block = BlockExample()
first_block.drop()

assert first_block.y == 1
assert second_block.y == 0
print(first_block.y, second_block.y)

### Easy Task 1: create and move a piece

**Student work:** complete `Piece.__init__` and `Piece.move`.

Constructor contract:

- store `kind`, `x`, and `y` as instance attributes;
- initialize `rotation` to zero;
- valid shape names are supplied by the caller.

Movement contract:

- add `dx` to the current x;
- add `dy` to the current y;
- return `None`.

The remaining methods are supplied. `cells()` converts local shape coordinates into board coordinates. `rotate()` selects the next stored orientation. Board collision checks are introduced later.

In [ ]:
class Piece:
    CELL_COUNT = 4  # An immutable class attribute shared by all pieces.

    def __init__(self, kind, x=3, y=0):
        # TODO: initialize the four instance attributes.
        raise NotImplementedError("Complete Piece.__init__.")

    def move(self, dx, dy):
        # TODO: change this piece's position.
        raise NotImplementedError("Complete Piece.move.")

    def cells(self):
        occupied = []
        shape = SHAPES[self.kind][self.rotation]
        for local_x, local_y in shape:
            occupied.append((self.x + local_x, self.y + local_y))
        return occupied

    def rotate(self):
        states = len(SHAPES[self.kind])
        self.rotation = (self.rotation + 1) % states

    def __str__(self):
        return f"{self.kind} piece at ({self.x}, {self.y})"

In [ ]:
# Validation: do not modify this cell.
first_piece = Piece("T", x=3, y=0)
second_piece = Piece("T", x=3, y=0)

assert first_piece.kind == "T"
assert (first_piece.x, first_piece.y, first_piece.rotation) == (3, 0, 0)
assert first_piece.move(-1, 2) is None
assert (first_piece.x, first_piece.y) == (2, 2)
assert (second_piece.x, second_piece.y) == (3, 0)
assert first_piece.cells() == [(3, 2), (2, 3), (3, 3), (4, 3)]
assert Piece.CELL_COUNT == 4
assert str(first_piece) == "T piece at (2, 2)"

print("Module 8 piece checks passed.")

### Module 8 checkpoint

Why does moving `first_piece` leave `second_piece` unchanged?

**Your answer:**

`CELL_COUNT` belongs to the class. Position belongs to each instance. Mutable game state, such as a board grid, must be created inside `__init__` rather than shared as a class attribute.

## 4. Module 9 review: reuse and specialize

Inheritance lets a child class reuse a parent class. `super().__init__()` initializes inherited state.

An overriding method uses the same name as a parent method and provides specialized behavior. In this short example, a faster demo block falls two cells per call.

In [ ]:
class FastBlockExample(BlockExample):
    def __init__(self, y=0):
        super().__init__(y)

    def drop(self):
        self.y += 2


demo_blocks = [BlockExample(), FastBlockExample()]

for block in demo_blocks:
    block.drop()  # Same method call, behavior supplied by each object.

assert [block.y for block in demo_blocks] == [1, 2]

### Easy Task 2: specialize the square piece

**Student work:** complete `SquarePiece`.

- In `__init__`, call the parent initializer with kind `"O"`, x, and y.
- Override `rotate()` so it makes no changes and returns `None`.
- Inherit `move()`, `cells()`, and `__str__()`.

This deliberately small subclass demonstrates specialization without creating seven separate shape classes.

In [ ]:
class SquarePiece(Piece):
    def __init__(self, x=3, y=0):
        # TODO: initialize the Piece parent for an O shape.
        raise NotImplementedError("Complete SquarePiece.__init__.")

    def rotate(self):
        # TODO: keep the square unchanged.
        raise NotImplementedError("Complete SquarePiece.rotate.")

In [ ]:
# Validation: do not modify this cell.
square = SquarePiece(x=4, y=2)
tee = Piece("T", x=4, y=2)
square_before = square.cells()
tee_before = tee.cells()

for piece in [square, tee]:
    piece.rotate()

assert isinstance(square, Piece)
assert square.kind == "O"
assert square.cells() == square_before
assert square.rotation == 0
assert tee.cells() != tee_before
assert tee.rotation == 1

square.move(-1, 1)
assert (square.x, square.y) == (3, 3)
print("Module 9 inheritance checks passed.")

### Module 9 checkpoint

The loop calls `rotate()` on both objects without selecting their concrete types. This is **polymorphism**.

Explain what is inherited, what is overridden, and why the same loop works for both pieces.

**Your answer:**

## 5. Final task, part A: protect the board

**Encapsulation:** `Board` keeps its grid in `_grid` and exposes methods for safe operations. The underscore marks a non-public detail; Python does not make it inaccessible.

**Abstraction:** callers use `can_place()` and `clear_lines()` without repeating their internal loops.

**Student work:** complete only `Board.can_place(cells)`.

For each `(x, y)`:

- reject a negative coordinate;
- reject x at or beyond width or y at or beyond height;
- reject an already occupied cell;
- return `True` only when every cell is valid.

Check boundaries before indexing. Negative Python indices otherwise select cells from the end of a list.

The constructor, settlement, copying, and row clearing are supplied.

In [ ]:
class Board:
    EMPTY = 0
    FILLED = 1

    def __init__(self, width=10, height=20):
        if width <= 0 or height <= 0:
            raise ValueError("Board dimensions must be positive.")
        self.width = width
        self.height = height
        self._grid = []
        for row in range(height):
            self._grid.append([self.EMPTY] * width)

    def can_place(self, cells):
        # TODO: check every coordinate and occupied cell without changing state.
        raise NotImplementedError("Complete Board.can_place.")

    def place(self, piece):
        cells = piece.cells()
        if not self.can_place(cells):
            raise ValueError("The piece cannot be placed here.")
        for x, y in cells:
            self._grid[y][x] = self.FILLED

    def clear_lines(self):
        remaining = []
        for row in self._grid:
            if self.EMPTY in row:
                remaining.append(row)

        cleared = self.height - len(remaining)
        for count in range(cleared):
            remaining.insert(0, [self.EMPTY] * self.width)

        self._grid = remaining
        return cleared

    def snapshot(self):
        copied_rows = []
        for row in self._grid:
            copied_rows.append(row.copy())
        return copied_rows

In [ ]:
# Validation: do not modify this cell.
board = Board(width=4, height=6)
other_board = Board(width=4, height=6)

assert board.can_place(SquarePiece(x=0, y=0).cells()) is True
assert board.can_place([(-1, 0)]) is False
assert board.can_place([(0, -1)]) is False
assert board.can_place([(4, 0)]) is False
assert board.can_place([(0, 6)]) is False

board.place(SquarePiece(x=0, y=4))
assert board.can_place(SquarePiece(x=0, y=4).cells()) is False
assert other_board.snapshot()[4][0] == Board.EMPTY

copy_of_board = board.snapshot()
copy_of_board[4][0] = Board.EMPTY
assert board.snapshot()[4][0] == Board.FILLED
assert board.snapshot()[0][0] == Board.EMPTY

board.place(SquarePiece(x=2, y=4))
assert board.clear_lines() == 2
assert board.snapshot() == [[0, 0, 0, 0] for row in range(6)]

print("Board checks passed.")

## 6. Final task, part B: coordinate objects

A `TetrisGame` **has a** `Board` and an active `Piece`. This is composition.

**Student work:** complete `TetrisGame.try_move(dx, dy)`.

1. If the game is over, return `False`.
2. Move the active piece by `dx` and `dy`.
3. Ask the board whether the new cells can be placed.
4. If valid, return `True`.
5. Otherwise restore the old position and return `False`.

A rejected move must leave the piece unchanged. The supplied timer, Down button, and Drop button all rely on this contract.

Other methods are supplied. A failed downward step locks the piece, clears full rows, awards 100 points per row, and spawns another piece. Rotation is rejected when the rotated cells do not fit; this small game does not shift a piece sideways to make a rotation fit.

In [ ]:
class TetrisGame:
    def __init__(self, width=10, height=20, seed=None):
        if width < 4 or height < 4:
            raise ValueError("The game needs at least a 4 by 4 board.")
        self._width = width
        self._height = height
        self._random = random.Random(seed)
        self.reset()

    @property
    def score(self):
        return self._score

    def reset(self):
        self.board = Board(self._width, self._height)
        self._score = 0
        self.lines = 0
        self.game_over = False
        self.active = self._new_piece()

    def _new_piece(self):
        kind = self._random.choice(list(SHAPES))
        if kind == "O":
            return SquarePiece(x=(self.board.width - 2) // 2, y=0)
        return Piece(kind, x=(self.board.width - 4) // 2, y=0)

    def try_move(self, dx, dy):
        # TODO: accept a valid move or undo an invalid move.
        raise NotImplementedError("Complete TetrisGame.try_move.")

    def try_rotate(self):
        if self.game_over:
            return False
        old_rotation = self.active.rotation
        self.active.rotate()
        if self.board.can_place(self.active.cells()):
            return True
        self.active.rotation = old_rotation
        return False

    def tick(self):
        if self.game_over:
            return False
        if self.try_move(0, 1):
            return True

        self.board.place(self.active)
        cleared = self.board.clear_lines()
        self.lines += cleared
        self._score += 100 * cleared
        self.active = self._new_piece()
        if not self.board.can_place(self.active.cells()):
            self.game_over = True
        return False

    def hard_drop(self):
        if self.game_over:
            return 0
        distance = 0
        while self.try_move(0, 1):
            distance += 1
        self.tick()
        return distance

In [ ]:
# Validation: do not modify this cell.
test_game = TetrisGame(width=4, height=6, seed=7)
test_game.active = SquarePiece(x=0, y=0)

assert test_game.try_move(-1, 0) is False
assert (test_game.active.x, test_game.active.y) == (0, 0)
assert test_game.try_move(1, 0) is True
assert (test_game.active.x, test_game.active.y) == (1, 0)

test_game.active = Piece("I", x=-1, y=0)
test_game.active.rotation = 1
assert test_game.board.can_place(test_game.active.cells()) is True
assert test_game.try_rotate() is False
assert test_game.active.rotation == 1

test_game.reset()
test_game.board.place(SquarePiece(x=2, y=4))
test_game.active = SquarePiece(x=0, y=4)
assert test_game.tick() is False
assert test_game.lines == 2
assert test_game.score == 200

test_game.reset()
test_game.active = SquarePiece(x=0, y=0)
assert test_game.hard_drop() == 4
assert sum(sum(row) for row in test_game.board.snapshot()) == 4

test_game.reset()
assert test_game.score == 0
assert test_game.lines == 0
assert test_game.game_over is False
assert sum(sum(row) for row in test_game.board.snapshot()) == 0
print("Final game checks passed.")

### A small Python class note

`game.score` uses the supplied `@property` getter. It reads `_score` without exposing a score setter. Game methods update the score.

`str(piece)` calls the supplied `__str__` method. These features make the interface convenient; you do not need to implement them in this practicum.

Stop here if any validation fails. Fix the failed method before creating the interface.

## 7. Supplied retro interface

The following helper code handles HTML display, button callbacks, and a cooperative timer. It is provided framework code, not assessed OOP material.

- Click **Start** for automatic falling at one step every 0.8 seconds.
- Click **Stop** to pause falling. Movement buttons still work for inspection.
- **Down** advances one step and locks a piece if it cannot descend.
- **Drop** settles the current piece immediately.
- **Restart** clears the board and score and leaves the game paused.
- Rerunning the launch cell closes the previous controls and cancels its timer.

Keep the runtime connected. Colab may respond more slowly than a desktop game because button events travel to the Python kernel.

In [ ]:
#@title Supplied retro interface: run unchanged
# Supplied framework. No TODOs in this cell.
import asyncio
from html import escape


def render_tetris(game, status="PAUSED"):
    grid = game.board.snapshot()

    if not game.game_over:
        for x, y in game.active.cells():
            if 0 <= x < game.board.width and 0 <= y < game.board.height:
                grid[y][x] = 2

    cells = []
    for y, row in enumerate(grid):
        for x, value in enumerate(row):
            if value == 2:
                color = "#e6ffec"
                label = "active"
            elif value == Board.FILLED:
                color = "#64c684"
                label = "settled"
            else:
                color = "#101b15"
                label = "empty"
            cells.append(
                f'<span title="Row {y + 1}, column {x + 1}: {label}" '
                f'style="display:block;width:22px;height:22px;'
                f'box-sizing:border-box;border:1px solid #263f2f;'
                f'background:{color}"></span>'
            )

    return (
        '<div style="background:#090f0b;color:#ddf5e3;'
        'font-family:monospace;padding:16px;width:max-content;'
        'border:2px solid #40664c">'
        '<div style="font-size:20px;letter-spacing:3px">TSI BLOCKS</div>'
        f'<div style="margin:10px 0">SCORE {game.score:04d} | LINES {game.lines}</div>'
        f'<div style="display:grid;grid-template-columns:repeat({game.board.width},22px);'
        'gap:1px;border:1px solid #40664c">'
        + "".join(cells)
        + '</div>'
        + f'<div style="margin-top:10px">{escape(status)}</div>'
        + '<div style="font-size:11px;margin-top:8px">'
        'LIGHT = ACTIVE | GREEN = SETTLED</div></div>'
    )


def show_tetris(game, fall_seconds=0.8):
    """Create notebook controls; return handles used for stopping and testing."""
    global _week12_view

    if fall_seconds <= 0:
        raise ValueError("The fall interval must be positive.")

    if "_week12_view" in globals():
        _week12_view["close"]()

    state = {"running": False, "task": None, "closed": False, "error": ""}
    screen = widgets.HTML()

    def refresh():
        if state["closed"]:
            return
        if state["error"]:
            status = "ERROR: " + state["error"]
        elif game.game_over:
            status = "GAME OVER - CLICK RESTART"
        elif state["running"]:
            status = "RUNNING"
        else:
            status = "PAUSED - START OR USE BUTTONS"
        screen.value = render_tetris(game, status)

    def stop():
        state["running"] = False
        task = state["task"]
        state["task"] = None
        if task is not None and not task.done():
            task.cancel()
        refresh()

    async def fall():
        try:
            while state["running"] and not game.game_over:
                await asyncio.sleep(fall_seconds)
                if not state["running"] or state["closed"]:
                    break
                game.tick()
                refresh()
        except asyncio.CancelledError:
            pass
        except Exception as error:
            # At the interface boundary, display an unexpected error visibly.
            state["error"] = str(error)
        finally:
            if state["task"] is asyncio.current_task():
                state["running"] = False
                state["task"] = None
                refresh()

    def start():
        if state["closed"] or state["running"] or game.game_over:
            return
        loop = asyncio.get_running_loop()
        state["error"] = ""
        state["running"] = True
        state["task"] = loop.create_task(fall())
        refresh()

    def restart():
        if state["closed"]:
            return
        stop()
        game.reset()
        state["error"] = ""
        refresh()

    def perform(operation):
        if state["closed"] or game.game_over:
            return
        try:
            operation()
        except Exception as error:
            state["error"] = str(error)
            stop()
            return
        if game.game_over:
            stop()
        else:
            refresh()

    def make_button(label, callback):
        button = widgets.Button(
            description=label,
            layout=widgets.Layout(width="78px", height="32px"),
        )
        button.on_click(lambda clicked: callback())
        return button

    buttons = {
        "start": make_button("Start", start),
        "stop": make_button("Stop", stop),
        "restart": make_button("Restart", restart),
        "left": make_button("Left", lambda: perform(lambda: game.try_move(-1, 0))),
        "right": make_button("Right", lambda: perform(lambda: game.try_move(1, 0))),
        "rotate": make_button("Rotate", lambda: perform(game.try_rotate)),
        "down": make_button("Down", lambda: perform(game.tick)),
        "drop": make_button("Drop", lambda: perform(game.hard_drop)),
    }

    top_controls = widgets.HBox([
        buttons["start"], buttons["stop"], buttons["restart"]
    ])
    move_controls = widgets.HBox([
        buttons["left"], buttons["right"], buttons["rotate"]
    ])
    drop_controls = widgets.HBox([buttons["down"], buttons["drop"]])
    panel = widgets.VBox([screen, top_controls, move_controls, drop_controls])

    def close():
        if state["closed"]:
            return
        stop()
        state["closed"] = True
        for button in buttons.values():
            button.disabled = True
            button.close()
        screen.close()
        top_controls.close()
        move_controls.close()
        drop_controls.close()
        panel.close()

    _week12_view = {
        "game": game,
        "state": state,
        "screen": screen,
        "buttons": buttons,
        "refresh": refresh,
        "start": start,
        "stop": stop,
        "restart": restart,
        "close": close,
    }

    refresh()
    display(panel)
    return _week12_view

### Launch your completed game

Run this cell after all four validation cells pass. Then click **Start**. Use **Stop** before rerunning class definitions or tests.

The interface uses your completed classes directly. It does not load an instructor implementation.

In [ ]:
game = TetrisGame()
controls = show_tetris(game)

## 8. Add two checks

**Student work:** add two meaningful assertions. Choose cases beyond the supplied validation, such as a blocked rotation, a failed placement leaving the board unchanged, or a new game having an independent board.

Stop the game before testing. Tests should create their own small objects rather than depend on the current live game.

In [ ]:
# TODO: write one assertion using a new test object.
# TODO: write a second assertion for a different behavior.

## 9. Explain your design

1. Which attributes belong to one piece rather than the class?
2. How does `Board` encapsulate its grid?
3. What does `SquarePiece` inherit, and what does it override?
4. Where does one method call produce object-specific behavior?
5. Why does `TetrisGame` contain a board instead of inheriting from `Board`?
6. What state must remain unchanged after a rejected move?

**Your answers:**

## Submission

Submit the completed notebook, two additional tests, and your explanations. Stop the timer, restart the runtime, and run all cells once more. Check the controls manually, then stop the game before saving.

The simple game has fixed speed and basic rotation. Its purpose is to make object responsibilities visible.

## References

- [Python tutorial: classes](https://docs.python.org/3.10/tutorial/classes.html)
- [Jupyter Widgets: button events](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20Events.html)
- [Jupyter Widgets: asynchronous interaction](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20Asynchronous.html)